In [2]:
# ==========================================
# CELL 1: IMPORTS
# ==========================================
import os
import random
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.layers import (
    Input, GlobalAveragePooling2D, BatchNormalization,
    Dense, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

print("TensorFlow version:", tf.__version__)

2026-04-27 20:23:52.160948: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777301632.182138  439578 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777301632.189442  439578 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777301632.233516  439578 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777301632.233553  439578 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777301632.233557  439578 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.1


In [3]:
# ==========================================
# CELL 2: REPRODUCIBILITY
# ==========================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print("Seed fixed:", SEED)

Seed fixed: 42


In [6]:
# ==========================================
# CELL 3: GPU SETUP (OPTIONAL)
# ==========================================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print("GPU setup error:", e)
else:
    print("No GPU found, running on CPU.")

GPU memory growth enabled.


In [4]:
# ==========================================
# CELL 4: CONFIGURATION
# ==========================================
img_size = 224
batch_size = 32
num_classes = 7


train_path = "/home/22EC1102/soumen/satarupa/hydrophobicity/Hydrophobicity Classes Photos/train"
val_path   = "/home/22EC1102/soumen/satarupa/hydrophobicity/Hydrophobicity Classes Photos/validation"



best_model_path = os.path.join( "teacher_model_mobilenetv3.keras")

print("Image size:", img_size)
print("Batch size:", batch_size)
print("Classes:", num_classes)
print("Best model path:", best_model_path)

Image size: 224
Batch size: 32
Classes: 7
Best model path: teacher_model_mobilenetv3.keras


In [5]:
# ==========================================
# CELL 5: DATA AUGMENTATION
# ==========================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_data = val_datagen.flow_from_directory(
    val_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

print("Class indices:", train_data.class_indices)

Found 2800 images belonging to 7 classes.
Found 700 images belonging to 7 classes.
Class indices: {'HC1': 0, 'HC2': 1, 'HC3': 2, 'HC4': 3, 'HC5': 4, 'HC6': 5, 'HC7': 6}


In [6]:
# ==========================================
# CELL 6: CHECK INPUT SHAPES
# ==========================================
images, labels = train_data[0]

print("Image batch shape :", images.shape)
print("Label batch shape :", labels.shape)
print("Single image shape:", images[0].shape)
print("Single label      :", labels[0])

Image batch shape : (32, 224, 224, 3)
Label batch shape : (32, 7)
Single image shape: (224, 224, 3)
Single label      : [0. 0. 1. 0. 0. 0. 0.]


In [7]:
# ==========================================
# CELL 7: BUILD TEACHER MODEL
# ==========================================
def build_teacher_model(num_classes=7, input_shape=(224, 224, 3)):
    inputs = Input(shape=input_shape)

    base_model = MobileNetV3Large(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = False

    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = Model(inputs, outputs, name="teacher_mobilenetv3_hydrophobicity")
    return model, base_model

teacher_model, base_model = build_teacher_model(num_classes=num_classes)
teacher_model.summary()

I0000 00:00:1777301659.658815  439578 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5900 MB memory:  -> device: 0, name: Tesla V100-PCIE-32GB, pci bus id: 0000:37:00.0, compute capability: 7.0


Model: "teacher_mobilenetv3_hydrophobicity"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 960)            │         3,840 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 960)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       246,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,249,031 (12.39 MB)

 Trainable params: 250,247 (977.53 KB)

 Non-trainable params: 2,998,784 (11.44 MB)

In [8]:
# ==========================================
# CELL 8: COMPILE MODEL
# ==========================================
teacher_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully.")

Model compiled successfully.


In [9]:
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=6,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=best_model_path,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    )
]

In [10]:
# ==========================================
# CELL 10: TRAIN TOP CLASSIFIER FIRST
# ==========================================
history_1 = teacher_model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20


I0000 00:00:1777301680.619341  439991 service.cc:152] XLA service 0x7f9660004f40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777301680.619380  439991 service.cc:160]   StreamExecutor device (0): Tesla V100-PCIE-32GB, Compute Capability 7.0
2026-04-27 20:24:40.864905: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777301682.147153  439991 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777301697.983451  439991 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 587ms/step - accuracy: 0.6124 - loss: 1.0988
Epoch 1: val_accuracy improved from None to 0.88571, saving model to teacher_model_mobilenetv3.keras
88/88 ━━━━━━━━━━━━━━━━━━━━ 96s 837ms/step - accuracy: 0.7425 - loss: 0.6929 - val_accuracy: 0.8857 - val_loss: 0.3213 - learning_rate: 0.0010
Epoch 2/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 369ms/step - accuracy: 0.8627 - loss: 0.3554
Epoch 2: val_accuracy improved from 0.88571 to 0.94714, saving model to teacher_model_mobilenetv3.keras
88/88 ━━━━━━━━━━━━━━━━━━━━ 35s 392ms/step - accuracy: 0.8714 - loss: 0.3393 - val_accuracy: 0.9471 - val_loss: 0.1508 - learning_rate: 0.0010
Epoch 3/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 0s 370ms/step - accuracy: 0.8799 - loss: 0.3144
Epoch 3: val_accuracy improved from 0.94714 to 0.95143, saving model to teacher_model_mobilenetv3.keras
88/88 ━━━━━━━━━━━━━━━━━━━━ 35s 395ms/step - accuracy: 0.8896 - loss: 0.2917 - val_accuracy: 0.9514 - val_loss: 0.1200 - learning_rate: 0.0010
Epoch 4/20
88/88 ━

In [11]:
# ==========================================
# CELL 13: EVALUATE MODEL
# ==========================================
val_loss, val_acc = teacher_model.evaluate(val_data, verbose=1)

print(f"Validation Loss    : {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - accuracy: 0.9786 - loss: 0.0704
Validation Loss    : 0.0704
Validation Accuracy: 0.9786


In [13]:
# ==========================================
# CELL 16: CHECK MODEL FILE SIZE
# ==========================================
def get_file_size_mb(path):
    size_bytes = os.path.getsize(path)
    size_mb = size_bytes / (1024 * 1024)
    return size_bytes, size_mb

for path in [best_model_path]:
    if os.path.exists(path):
        size_bytes, size_mb = get_file_size_mb(path)
        print(f"{path}")
        print(f"  Size: {size_bytes} bytes")
        print(f"  Size: {size_mb:.2f} MB")
    else:
        print(f"{path} not found")

teacher_model_mobilenetv3.keras
  Size: 15705913 bytes
  Size: 14.98 MB
